# Stage 11 — paper-comparable headline run (Kaggle T4 / L4)

Single training run on the full 122-signer dataset under the paper's 8:1:1 split.  Hyperparameters frozen from Stage 9a's ablation winners (λ_ctc=0.5, decoder=2 layers).  Test-set evaluation runs **exactly once** at the end.

## Prerequisites (attach as datasets)
1. `wita-full-english-landmark-cache` — produced by the `extract_landmarks_122_kaggle.ipynb` kernel.
2. (Optional, for sanity check 1) the 38-signer `skeleton_features_t32.pt` cache.

## Wall-clock
| Phase | Time |
|---|---|
| Sanity checks (8 of them) | <2 min |
| Training: 80 epochs × ~280 steps/epoch | ~2 h on L4 / ~3 h on T4 |
| Final test eval (once!) | ~3 min |
| Total | **~2 – 3 h** |

## Outputs
- `/kaggle/working/checkpoints/stage11_best.pt`
- `/kaggle/working/logs/stage11_training.json` + `_full.json`
- `/kaggle/working/logs/stage11_test_headline.json`  ← the three numbers
- `/kaggle/working/logs/stage11_test_full.json`
- `/kaggle/working/logs/.stage11_test_evaluated` (marker — gates re-runs)

## Cell 1 — Install + clone

In [ ]:
%%capture
!pip install editdistance huggingface_hub hgtk scipy --quiet
!pip install 'mediapipe==0.10.14' --quiet

import sys, os, glob
!rm -rf /kaggle/working/wita_v2
!git clone -b iterative-ablation "https://github.com/Gaurs86/WiTA-v2.git" '/kaggle/working/wita_v2'
sys.path.insert(0, '/kaggle/working')
import torch
print(f'GPU : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none"}')

## Cell 2 — Locate the landmark cache

In [ ]:
import os, glob, json
def _find_dir(name):
    cs = (glob.glob(f'/kaggle/working/**/{name}', recursive=True)
          + glob.glob(f'/kaggle/input/**/{name}', recursive=True))
    return cs[0] if cs else None

CACHE_ROOT = _find_dir('landmark_cache_122')
assert CACHE_ROOT, ('landmark_cache_122/ not found. Attach the '
                    'wita-full-english-landmark-cache dataset.')
print(f'cache_root : {CACHE_ROOT}')

OLD_CACHE = _find_dir('skeleton_features_t32.pt') or None
print(f'old 38-signer cache (for sanity check 1) : {OLD_CACHE}')

## Cell 3 — Run all 8 sanity checks  (fail-fast pre-launch)

If anything fails, **stop**.  Do not start the 2-3 h training run on a contaminated cache.

In [ ]:
extra = f' --old-cache {OLD_CACHE}' if OLD_CACHE else ''
!python /kaggle/working/wita_v2/scripts/sanity_check_stage11.py --cache-root {CACHE_ROOT}{extra}

## Cell 4 — Config

In [ ]:
import logging, random
import numpy as np
from wita_v2.configs.default import Config, DataConfig, EncoderConfig, TrainConfig

LOG_DIR  = '/kaggle/working/logs'
CKPT_DIR = '/kaggle/working/checkpoints'
os.makedirs(LOG_DIR, exist_ok=True); os.makedirs(CKPT_DIR, exist_ok=True)
logging.basicConfig(level=logging.INFO,
    format='%(asctime)s  %(levelname)-7s  %(name)s — %(message)s',
    handlers=[logging.StreamHandler(),
              logging.FileHandler(os.path.join(LOG_DIR, 'stage11.log'))])

SEED         = 42
VARIANT_NAME = 'stage11'
NUM_EPOCHS   = 80
BATCH_SIZE   = 32
LAMBDA_CTC   = 0.5     # Stage 9a winner
DEC_N_LAYERS = 2       # Stage 9a winner
DEC_N_HEADS  = 4
LR_PEAK      = 5e-4
WEIGHT_DECAY = 5e-2
GRAD_CLIP    = 1.0
DROPOUT      = 0.2
WARMUP_PCT   = 0.05
D_MODEL      = 256
N_LAYERS     = 4
N_HEADS      = 4
CONV_KERNEL  = 15
UPSAMPLE     = 2

cfg = Config(
    data=DataConfig(hf_repo_id='yewon816/WiTA', lang='english',
                    max_zips=None, max_frames=64, seed=SEED),
    encoder=EncoderConfig(arch='siglip'),
    train=TrainConfig(num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE, lr=LR_PEAK,
                      weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
                      num_workers=2, warmup_pct=WARMUP_PCT, seed=SEED,
                      checkpoint_dir=CKPT_DIR),
).build()
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
torch.backends.cudnn.benchmark = True
print(f'Device       : {cfg.device}')
print(f'Variant      : {VARIANT_NAME}')
print(f'lambda_ctc   : {LAMBDA_CTC}')
print(f'dec_n_layers : {DEC_N_LAYERS}')

## Cell 5 — Train  (val checkpoint selection on val_overall_cer)

In [ ]:
from wita_v2.training.stage11_train  import train_stage11
from wita_v2.datasets.skeleton_augment import LandmarkAugment

train_aug = LandmarkAugment()    # Stage 1 v2 defaults
result = train_stage11(
    cache_root=CACHE_ROOT, cfg=cfg,
    num_epochs=NUM_EPOCHS, batch_size=BATCH_SIZE,
    lr_peak=LR_PEAK, weight_decay=WEIGHT_DECAY, grad_clip=GRAD_CLIP,
    dropout=DROPOUT, d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    conv_kernel=CONV_KERNEL, upsample=UPSAMPLE, warmup_pct=WARMUP_PCT,
    dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
    lambda_ctc=LAMBDA_CTC, label_smoothing=0.1,
    transform=train_aug, seed=SEED,
    checkpoint_dir=CKPT_DIR, log_dir=LOG_DIR, variant=VARIANT_NAME,
)
print('\n=== Best on validation ===')
print(json.dumps(result['best_payload'], indent=2, default=str))

## Cell 6 — Test evaluation  (EXACTLY ONCE — DO NOT RE-RUN)

Per §7 of the Stage 11 prompt, this cell must run exactly once.  A marker file in `/kaggle/working/logs/.stage11_test_evaluated` is created when it succeeds; re-running will raise a RuntimeError unless the marker is deleted by hand.

**Stop and write the three CER numbers down before doing anything else.**

In [ ]:
from wita_v2.training.stage11_train import final_test_eval
ckpt = result['checkpoint_path']
test_out = final_test_eval(
    cache_root=CACHE_ROOT, checkpoint=ckpt, cfg=cfg,
    batch_size=BATCH_SIZE,
    d_model=D_MODEL, n_layers=N_LAYERS, n_heads=N_HEADS,
    conv_kernel=CONV_KERNEL, dropout=DROPOUT, upsample=UPSAMPLE,
    dec_n_layers=DEC_N_LAYERS, dec_n_heads=DEC_N_HEADS,
    log_dir=LOG_DIR, variant=VARIANT_NAME,
)

## Cell 7 — Band classification (only after Cell 6 wrote the headline)

In [ ]:
h = test_out['headline']
ov, lex, non = h['test_overall_cer'], h['test_lex_cer'], h['test_nonlex_cer']
p = h['paper_baseline']
print(f'\nyour overall = {ov:.4f}   paper = {p["overall"]}')
print(f'your lex     = {lex:.4f}   paper = {p["lex"]}')
print(f'your nonlex  = {non:.4f}   paper = {p["nonlex"]}')
if ov <= 0.29 and lex <= 0.27:
    band = 'STRONG_PASS'
    next_step = 'write the thesis; optionally chain Stage 9b LM rescoring'
elif 0.29 < ov <= 0.32 and lex <= 0.27:
    band = 'PASS_ON_LEX'
    next_step = 'add KenLM (Stage 9b) for a small overall lift'
elif ov <= 0.35:
    band = 'COMPETITIVE'
    next_step = 'Stage 9b first; consider Stage 10 partial unfreeze'
else:
    band = 'UNDERPERFORMS'
    next_step = 'diagnose: train-val NLL gap; per-signer scatter; HRNet swap on tail'
print(f'\nBand: {band}')
print(f'Recommended next step: {next_step}')

## Cell 8 — Diagnostics figures (read but do NOT use to override Cell 6)

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# Length-bucketed test CER.
bucket_cer = test_out['per_length_cer']
bucket_order = ['1-4','5-8','9-12','13-inf']
bs = [b for b in bucket_order if b in bucket_cer]
ys = [bucket_cer[b] for b in bs]
fig, ax = plt.subplots(figsize=(7, 3.2))
ax.bar(bs, ys, color='#1f77b4')
for x, y in zip(bs, ys): ax.text(x, y+0.005, f'{y:.3f}', ha='center', fontsize=8)
ax.set_xlabel('label length bucket'); ax.set_ylabel('test CER')
ax.set_title('Stage 11 — length-bucketed test CER')
ax.grid(True, linestyle=':', alpha=0.4, axis='y')
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage11_length_buckets.png'), dpi=140)
plt.show()

# Per-signer test CER scatter.
per_signer = test_out['per_signer_cer']
items = sorted(per_signer.items(), key=lambda kv: kv[1])
fig, ax = plt.subplots(figsize=(14, 4.0))
ax.scatter(range(len(items)), [v for _, v in items], s=18, color='#2ca02c')
ax.axhline(0.55, color='green', linestyle='--', alpha=0.4, label='easy ≤ 0.55')
ax.axhline(0.75, color='red',   linestyle='--', alpha=0.4, label='hard ≥ 0.75')
ax.set_xlabel('signer (sorted by CER)'); ax.set_ylabel('test CER')
ax.set_title(f'Stage 11 — per-signer test CER  (n={len(items)} test signers)')
ax.set_xticks([]); ax.grid(True, linestyle=':', alpha=0.3)
ax.legend(frameon=False)
plt.tight_layout()
plt.savefig(os.path.join(LOG_DIR, 'stage11_per_signer_test.png'), dpi=140)
plt.show()

## Cell 9 — Commit kernel

Save Version → Save & Run All to preserve:
- `logs/stage11_test_headline.json`  ← the three CER numbers
- `logs/stage11_test_full.json`
- `logs/stage11_training.json` + `_full.json`
- `logs/stage11_length_buckets.png`, `stage11_per_signer_test.png`
- `checkpoints/stage11_best.pt`
- `logs/.stage11_test_evaluated`  ← marker, prevents accidental re-eval

Send `stage11_test_headline.json` here for the writeup.